In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .option("mode", "dropMalformed")\
        .load("abfss://bronze@storagefornetflixproject.dfs.core.windows.net/netflix_titles")

In [0]:
df.display()

In [0]:
df = df.fillna({'duration_minutes': 0, 'duration_seasons': 1})

In [0]:
df.display()

In [0]:
df = df.withColumn("short_title",split(col("rating"),'-')[0])

In [0]:
df = df.drop("_rescued_data")

In [0]:
df = df.withColumn("type_Int",when(col("type")=="Movie",1).otherwise(0))

In [0]:
df = df.withColumn("rank_wise",dense_rank().over(Window.orderBy(col("release_year").desc())))

In [0]:
df.display()

In [0]:

df.groupBy("release_year").agg(count("title").alias("total_count")).orderBy(col("release_year").asc()).display()

In [0]:
df.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@storagefornetflixproject.dfs.core.windows.net/netflix_titles")\
    .saveAsTable("netflix_cata.silver.netflix_titles")